# Extract dataset from djimaging

- same dataset as used for Helsinki poster figure 6 and 7

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datajoint as dj
import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

/workspace/.venv/lib/python3.12/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Access to djimaging database

## Define paths

In [3]:
username = !whoami
username = username[0]
username

'fsalomon'

In [4]:
home_directory = os.path.expanduser("~")
home_directory

'/gpfs01/euler/User/fsalomon'

In [5]:
# Path to djimaging
path_to_djimaging = f'{home_directory}/github/eulerlab/'

# Clone djimaging if you haven't downloaded it yet
# assert os.isdir(path_to_djimaging), 'Create target folder before cloning djimaging'
# !cd {path_to_djimaging} && git clone git@github.com:eulerlab/djimaging.git

# Install djimaging if not done yet
# assert os.isdir(os.path.join(path_to_djimaging, 'djimaging')), 'Create target folder before cloning djimaging'
# !cd {path_to_djimaging}/djimaging/ && sudo pip install -e .

## Prepare dj config

In [6]:
# Set config file
config_file = f'{home_directory}/GitHub/docker/dj_{username}_conf.json'
assert os.path.isfile(config_file), f'Set the path to your config file: {config_file}'

In [7]:
# Define a schema name or use the default name for your personal test schema
# It should start with ageuler and have some meaningful name after that
schema_name = f"ageuler_{username}_neuromodulation_ac"

In [8]:
# Do you want to use the RGC classifier?
# If so, make sure to add the respective tables into your schema.
use_rgc_classifier = False

if use_rgc_classifier:  # Define any existing outputfolder for the classifier to be saved
    output_folder = f'{home_directory}/datajoint/rgc_classifier'
    assert os.path.isdir(output_folder), f'Set path to output directory: {output_folder}'

In [9]:
# Load configuration for user
dj.config.load(config_file)
dj.config['schema_name'] = schema_name

if use_rgc_classifier:
    from djimaging.tables.classifier.rgc_classifier import prepare_dj_config_rgc_classifier
    prepare_dj_config_rgc_classifier(output_folder)

print("schema_name:", dj.config['schema_name'])
dj.conn()

[2026-10-02 14:03:36,167][INFO]: DataJoint is configured from /gpfs01/euler/User/fsalomon/GitHub/docker/dj_fsalomon_conf.json
[2026-10-02 14:03:36,229][INFO]: DataJoint 0.14.6 connected to fsalomon@172.25.240.200:3306


schema_name: ageuler_fsalomon_neuromodulation_ac


DataJoint connection (connected) fsalomon@172.25.240.200:3306

## Create or load schema

In [10]:
# Import your own schema, which defines the tables you will have in your database and how they are connected.
from djimaging.user.amacrine_cells_neuromodulation.schemas.amacrine_cells_neuromodulation_schema import *

In [11]:
if use_rgc_classifier:
    try:
        CelltypeAssignment()
    except Exception as e:
        import warnings
        warnings.warn("""
            If you want to include the RGC classifier in your database, you have to copy the respective tables from 
            djimaging/schemas/full_rgc_schema.py 
            to the schema you just imported (see above).
            Then restart the notebook and try again.
            """)

## Important note

If the schema with the name `schema_name = f"ageuler_{username}_test"` already exists, it is important that the schema definition here is the same as it was when the schema was created.
If you did the other tutorial first, and did not delete (=drop) the schema afterwards, this will not be the case, for example.
Then you already have a schema with the same name but different tables, the first being based on the schema `rgc_classifier_schema`, and this one being based on `my_schema`. This can result in a variety of problems, so you either have to change the schema name here or drop the old schema first.

Outside of this tutorial, in most cases, you want exactly one schema per project to never run into this problem.

In [12]:
from djimaging.utils.dj_utils import activate_schema

activate_schema(schema=schema, create_schema=True, create_tables=True)
schema

Schema `ageuler_fsalomon_neuromodulation_ac`

# Chirp dataset

## For selected target dates

In [13]:
target_dates = [
    datetime.date(2025, 9, 8),
    datetime.date(2025, 10, 23),
    datetime.date(2025, 11, 19)]

In [14]:
chirp_qi = ChirpQI & {"stim_name": "gChirp"} & [{"date": d} for d in target_dates]
chirp_qi

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_id integer id of each ROI,preprocess_id unique param set id,qidx quality index as signal to noise ratio,min_qidx minimum quality index as 1/r (r = #repetitions)
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,1,1,0.205932,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,2,1,0.26708,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,3,1,0.23731,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,4,1,0.215393,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,5,1,0.223126,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,6,1,0.194005,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,7,1,0.279805,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,8,1,0.26186,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,9,1,0.24477,0.2
Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,10,1,0.207242,0.2


In [15]:
df = pd.DataFrame(chirp_qi.fetch(as_dict = True))
df

,experimenter,date,exp_num,raw_id,field,region,cond1,stim_name,cond2,roi_id,preprocess_id,qidx,min_qidx
0,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,1,1,0.205932,0.2
1,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,2,1,0.267080,0.2
2,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,3,1,0.237310,0.2
3,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,4,1,0.215393,0.2
4,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,5,1,0.223126,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5883,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,DETANO,183,1,0.486900,0.2
5884,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,DETANO,184,1,0.573505,0.2
5885,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,DETANO,185,1,0.402241,0.2
5886,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,DETANO,186,1,0.441575,0.2


In [16]:
df_c1 = df[df["cond2"] == "C1"]
df_c1

,experimenter,date,exp_num,raw_id,field,region,cond1,stim_name,cond2,roi_id,preprocess_id,qidx,min_qidx
0,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,1,1,0.205932,0.2
1,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,2,1,0.267080,0.2
2,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,3,1,0.237310,0.2
3,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,4,1,0.215393,0.2
4,Salomon,2025-09-08,2,1,XY1,RR,v,gChirp,C1,5,1,0.223126,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5696,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,C1,183,1,0.248336,0.2
5697,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,C1,184,1,0.208375,0.2
5698,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,C1,185,1,0.222073,0.2
5699,Salomon,2025-11-19,2,1,XY5,RR,t,gChirp,C1,186,1,0.206722,0.2


In [17]:
dates = df["date"].unique()

for date in dates:

    df_date = df_c1[df_c1["date"] == date]
    fields = df_date["field"].unique()

    for field in fields:

        df_field = df_date[df_date["field"] == field]
        cond2 = df_field["cond2"].unique()

        df_field_filtered = df_field[df_field["qidx"] > 0.35]

        print(f"{date}; {field}: Filtered: {len(df_field_filtered)}, All: {len(df_field)}, In %: {np.round(len(df_field_filtered)/len(df_field)*100, 0)}")

2025-09-08; XY1: Filtered: 6, All: 93, In %: 6.0
2025-09-08; XY2: Filtered: 61, All: 187, In %: 33.0
2025-09-08; XY3: Filtered: 84, All: 134, In %: 63.0
2025-09-08; XY4: Filtered: 8, All: 160, In %: 5.0
2025-09-08; XY6: Filtered: 28, All: 149, In %: 19.0
2025-09-08; XY7: Filtered: 246, All: 249, In %: 99.0
2025-09-08; XY8: Filtered: 221, All: 258, In %: 86.0
2025-10-23; XY7: Filtered: 147, All: 177, In %: 83.0
2025-10-23; XY8: Filtered: 13, All: 183, In %: 7.0
2025-10-23; XY1: Filtered: 1, All: 167, In %: 1.0
2025-10-23; XY2: Filtered: 71, All: 163, In %: 44.0
2025-10-23; XY3: Filtered: 25, All: 113, In %: 22.0
2025-10-23; XY4: Filtered: 189, All: 189, In %: 100.0
2025-10-23; XY5: Filtered: 177, All: 182, In %: 97.0
2025-10-23; XY6: Filtered: 154, All: 166, In %: 93.0
2025-11-19; XY4: Filtered: 187, All: 187, In %: 100.0
2025-11-19; XY5: Filtered: 0, All: 187, In %: 0.0


In [18]:
# Quality threshold for individual recordings
qidx_threshold = 0.35

# Minimum fraction of recordings that need to pass
quality_threshold = 0.30

# Calculate quality for each date-field pair
field_quality = (
    df_c1
    .assign(passes_quality=df_c1["qidx"] > qidx_threshold)
    .groupby(["date", "field"])["passes_quality"]
    .mean()
)

# Keep only date-field pairs with >= 30% good recordings
good_fields = field_quality[field_quality >= quality_threshold].index

# Filter the original dataframe
df_c1_filtered = (
    df_c1.set_index(["date", "field"])
         .loc[good_fields]
         .reset_index()
)

print(f"Original rows: {len(df_c1)}")
print(f"Filtered rows: {len(df_c1_filtered)}")

Original rows: 2944
Filtered rows: 1892


In [19]:
# Get the date-field-roi combinations that survived the C1 filtering
keys = df_c1_filtered[["date", "field", "roi_id"]].drop_duplicates()

# Select all corresponding rows from the original dataframe
df_filtered = df.merge(
    keys,
    on=["date", "field", "roi_id"],
    how="inner"
)

df_filtered

,experimenter,date,exp_num,raw_id,field,region,cond1,stim_name,cond2,roi_id,preprocess_id,qidx,min_qidx
0,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,1,1,0.306987,0.2
1,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,2,1,0.397395,0.2
2,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,3,1,0.442943,0.2
3,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,4,1,0.267124,0.2
4,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,5,1,0.239614,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3779,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,183,1,0.331610,0.2
3780,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,184,1,0.448928,0.2
3781,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,185,1,0.294794,0.2
3782,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,186,1,0.336143,0.2


In [20]:
dates = df_filtered["date"].unique()

for date in dates:

    df_date = df_filtered[df_filtered["date"] == date]
    fields = df_date["field"].unique()

    for field in fields:

        df_field = df_date[df_date["field"] == field]
        cond2 = df_field["cond2"].unique()

        print(f"{date}; {field}: {cond2}")
    
    #print(f"{date}: {fields}")

2025-09-08; XY2: ['C1' 'DETANO']
2025-09-08; XY3: ['C1' 'C2']
2025-09-08; XY7: ['C1' 'C2']
2025-09-08; XY8: ['C1' 'DETANO']
2025-10-23; XY7: ['C1' 'C2']
2025-10-23; XY2: ['C1' 'DETANO']
2025-10-23; XY4: ['C1' 'DETANO']
2025-10-23; XY5: ['C1' 'C2']
2025-10-23; XY6: ['C1' 'DETANO']
2025-11-19; XY4: ['C1' 'C2']


In [21]:
averages = Averages & {"stim_name": "gChirp"} & [{"date": d} for d in target_dates]
df_averages = pd.DataFrame(averages.fetch(as_dict=True))

In [22]:
key_cols = [
    "experimenter", "date", "exp_num", "raw_id", "field",
    "region", "cond1", "stim_name", "cond2", "roi_id", "preprocess_id"
]

result = df_averages.merge(
    df_filtered,
    on=key_cols,
    how="inner"  # keeps only rows whose key combo exists in both dataframes
)

result

,experimenter,date,exp_num,raw_id,field,region,cond1,stim_name,cond2,roi_id,preprocess_id,average,average_norm,average_t0,average_dt,triggertimes_rel,qidx,min_qidx
0,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,1,1,"[0.23404275666265387, 0.2381303010000348, 0.24...","[0.13093252217697324, 0.13321925173543125, 0.1...",0.0,0.002,"[0.0, 4.9980016]",0.306987,0.2
1,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,2,1,"[0.8592742807976649, 0.862168467678105, 0.8650...","[0.48455278264264817, 0.48618484162281833, 0.4...",0.0,0.002,"[0.0, 4.9980016]",0.397395,0.2
2,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,3,1,"[0.9966721981509353, 0.9982692731988386, 0.999...","[0.501599497802753, 0.5024032646214401, 0.5032...",0.0,0.002,"[0.0, 4.9980016]",0.442943,0.2
3,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,4,1,"[-0.11012223462448449, -0.11653696637358568, -...","[-0.06841417883949284, -0.07239937407810096, -...",0.0,0.002,"[0.0, 4.9980016]",0.267124,0.2
4,Salomon,2025-09-08,2,1,XY2,RR,v,gChirp,C1,5,1,"[1.0140207452784273, 1.014220858071091, 1.0144...","[0.5124799892216649, 0.5125811250241669, 0.512...",0.0,0.002,"[0.0, 4.9980016]",0.239614,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3779,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,183,1,"[0.8112028982575517, 0.7867748997893866, 0.773...","[0.3587111864995656, 0.34790920794013624, 0.34...",0.0,0.002,"[0.0, 4.9971995]",0.331610,0.2
3780,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,184,1,"[1.2339049363468084, 1.2414545536000525, 1.245...","[0.5019784434913204, 0.5050497863525937, 0.506...",0.0,0.002,"[0.0, 4.9971995]",0.448928,0.2
3781,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,185,1,"[0.1381078234772653, 0.142473841751133, 0.1468...","[0.06722641006479439, 0.069351646184222, 0.071...",0.0,0.002,"[0.0, 4.9971995]",0.294794,0.2
3782,Salomon,2025-11-19,2,1,XY4,RR,t,gChirp,C2,186,1,"[0.4181908417071479, 0.4380852987568168, 0.457...","[0.15175416080599652, 0.15897351219575662, 0.1...",0.0,0.002,"[0.0, 4.9971995]",0.336143,0.2


In [23]:
df_previous = pd.read_pickle("../../../nitric_oxide_imaging/notebooks/helsinki_figures/datasets/figure7C_GlobalChirpAverages.pkl")

In [24]:
df_previous.equals(result)

True

In [25]:
result.to_pickle("../datasets/Averages_gChirp_C1C2DETANO.pkl")